## Standard Imports

In [5]:
!pip install earthengine-api pandas pillow numpy requests

  Using cached httplib2-0.31.2-py3-none-any.whl.metadata (2.2 kB)
  Using cached uritemplate-4.2.0-py3-none-any.whl.metadata (2.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 479.3/479.3 kB 135.2 kB/s eta 0:00:001m133.7 kB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.0/15.0 MB 78.4 kB/s eta 0:00:00m eta 0:00:010:00:05m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 240.6/240.6 kB 80.7 kB/s eta 0:00:00 kB/s eta 0:00:01:01
Using cached httplib2-0.31.2-py3-none-any.whl (91 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.5/324.5 kB 93.6 kB/s eta 0:00:0031m97.2 kB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 173.3/173.3 kB 118.4 kB/s eta 0:00:001m120.7 kB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 138.1 kB/s eta 0:00:00 kB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.8/122.8 kB 125.4 kB/s eta 0:00:001m117.5 kB/s eta 0:00:01
Using cached uritemplate-4.2.0-py3-none-any.whl (11 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [1]:
import ee
import os
from pathlib import Path
import requests
from PIL import Image
import numpy as np
from io import BytesIO


print('Imported')

Imported


In [7]:

# Initialize Earth Engine
try:
    ee.Initialize(project = 'ndvi-trend-481019')
except:
    ee.Authenticate()
    ee.Initialize(project = 'ndvi-trend-481019')

print('Done')

Done


## Hasdeo Forest .TIF Format Downloader

In [3]:
"""
Hasdeo Forest GeoTIFF Downloader — 1000–4000 IMAGE DATASET
===========================================================
Exports individual Sentinel-2 SR scenes (NOT composites) to Google Drive
for the Hasdeo Arand forest region, targeting 1000–4000 raw images.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
WHY THE OLD SCRIPT ONLY GAVE ~250 IMAGES
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  The previous script exported median COMPOSITES — one image per
  month/season per region. Many months had 0 cloud-free scenes
  and were skipped, giving only ~632 tasks of which GEE completed ~250.

  This new script exports INDIVIDUAL SCENES (each Sentinel-2 overpass)
  plus composites, so you get every available image instead of 1 per month.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
IMAGE COUNT STRATEGY (targeting 1000–4000)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Part 1 — Individual scenes (main bulk)
    Sentinel-2 revisits every ~5 days over India.
    10 years × ~70 passes/year × 7 regions = ~4900 potential
    (actual depends on cloud; 40% threshold used to maximise yield)

  Part 2 — Monthly composites  (same as before, kept as bonus)
    120 months × 7 regions = 840 potential

  Part 3 — Seasonal composites  (kept as bonus)
    10 years × 7 regions × 2 seasons = 140 potential

  TOTAL POTENTIAL                             ≈ 5880+
  Realistic yield (40% cloud, all scenes)     ≈ 1500–4000

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
OUTPUT
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Drive folder  : Hasdeo_1000_Dataset_AllBands_new/
  Format        : 12-band GeoTIFF (B1–B12 excl. B10, raw Uint16, 30 m)
  Bands         : B1,B2,B3,B4,B5,B6,B7,B8,B8A,B9,B11,B12
  File size     : ~100–300 MB per image at 30 m resolution
  Naming:
    Individual  : SCENE_<Region>_<YYYYMMDD>_<orbit>_<phase>.tif
    Monthly     : MON_<Region>_<YYYY>_<MM>_<phase>.tif
    Seasonal    : SEAS_<DRY|WET>_<Region>_<YYYY>_<phase>.tif

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
SETUP & USAGE
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  1. pip install earthengine-api tqdm
  2. earthengine authenticate        # one-time browser login
  3. python hasdeo_4000_downloader.py
  4. Monitor tasks: https://code.earthengine.google.com/tasks
  5. Files appear in Google Drive → syncs locally via desktop app
"""

import ee
import csv
import json
import time
from pathlib import Path
from datetime import datetime
from tqdm import tqdm

# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────

PROJECT_ID          = 'glacier-probe-model-475519'   # ← your GEE Cloud project ID
OUTPUT_DIR          = 'hasdeo_4000_local'            # local metadata/logs directory

# ── Cloud cover thresholds ──────────────────────────────────────────────────
# Individual scenes: 40% to capture maximum images (you can filter locally)
# Composites: 20% for clean median composites
MAX_CLOUD_INDIVIDUAL = 40
MAX_CLOUD_COMPOSITE  = 20

MIN_IMAGES_COMPOSITE = 1   # skip composite period if fewer scenes

SCALE = 30                 # 30 m resolution (set to 10 for native Sentinel-2 res)

# ── New folder name (appended with _new per your request) ──────────────────
DRIVE_FOLDER = 'Hasdeo_1000_Dataset_AllBands_new'

# ── 7 sub-regions covering Hasdeo Arand (WGS84: [west, south, east, north])
REGIONS = {
    'Hasdeo_Full':    [82.58, 22.40, 83.20, 22.90],  # entire block
    'Hasdeo_North':   [82.58, 22.65, 83.20, 22.90],  # northern half
    'Hasdeo_South':   [82.58, 22.40, 83.20, 22.65],  # southern half
    'Hasdeo_Core':    [82.75, 22.50, 83.05, 22.80],  # dense forest core
    'Hasdeo_East':    [82.95, 22.40, 83.20, 22.90],  # eastern edge
    'Hasdeo_West':    [82.58, 22.40, 82.90, 22.90],  # western edge
    'Hasdeo_Buffer':  [82.50, 22.30, 83.30, 23.00],  # extended buffer zone
}

# ── All 12 spectral bands + SCL for cloud masking ────────────────────────
# B10 excluded — cirrus band, not present in S2 SR (surface reflectance) product
S2_BANDS = ['B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7',
            'B8', 'B8A', 'B9', 'B11', 'B12', 'SCL']

# Output bands (no SCL in final export)
OUTPUT_BANDS = ['B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B9', 'B11', 'B12']

# ─────────────────────────────────────────────────────────────────────────────
# PHASE LABELLING
# ─────────────────────────────────────────────────────────────────────────────

def get_phase(year: int, month: int) -> str:
    if 2015 <= year <= 2017:
        return 'pre_event'
    if year == 2022 and month in (3, 4):
        return 'second_event'
    if 2018 <= year <= 2023:
        return 'first_event'
    if year == 2024:
        return 'post_event'
    return 'background'

# ─────────────────────────────────────────────────────────────────────────────
# TIME PERIOD BUILDER
# ─────────────────────────────────────────────────────────────────────────────

def build_monthly_periods():
    """Build monthly windows Jan 2015 – Dec 2024 (120 periods)."""
    periods = []
    for year in range(2015, 2025):
        for month in range(1, 13):
            start = f'{year}-{month:02d}-01'
            end   = (f'{year + 1}-01-01'
                     if month == 12
                     else f'{year}-{month + 1:02d}-01')
            periods.append((start, end, get_phase(year, month), year, month))
    return periods

TIME_PERIODS = build_monthly_periods()

# ─────────────────────────────────────────────────────────────────────────────
# EARTH ENGINE INITIALISATION
# ─────────────────────────────────────────────────────────────────────────────

def init_ee():
    try:
        ee.Initialize(project=PROJECT_ID)
        print(f"✓ Earth Engine initialised  |  project: {PROJECT_ID}")
    except Exception:
        print("⚠  Authenticating with Earth Engine (browser will open)...")
        ee.Authenticate()
        ee.Initialize(project=PROJECT_ID)
        print("✓ Authenticated and initialised.")

def create_local_dirs():
    for sub in ['metadata', 'logs']:
        Path(f"{OUTPUT_DIR}/{sub}").mkdir(parents=True, exist_ok=True)
    print(f"✓ Local dirs ready: {OUTPUT_DIR}/")

# ─────────────────────────────────────────────────────────────────────────────
# CLOUD MASKING
# ─────────────────────────────────────────────────────────────────────────────

def mask_s2_clouds(image):
    """
    Dual-layer cloud mask using QA60 and SCL bands.
    Applied to composites for clean median; individual scenes are exported
    with the mask applied but without aggressive filtering.
    """
    qa       = image.select('QA60')
    qa_mask  = (qa.bitwiseAnd(1 << 10).eq(0)
                  .And(qa.bitwiseAnd(1 << 11).eq(0)))
    scl      = image.select('SCL')
    scl_mask = (scl.neq(3) .And(scl.neq(8))
                            .And(scl.neq(9))
                            .And(scl.neq(10))
                            .And(scl.neq(11)))
    return image.updateMask(qa_mask.And(scl_mask))

# ─────────────────────────────────────────────────────────────────────────────
# EXPORT TO DRIVE
# ─────────────────────────────────────────────────────────────────────────────

def export_to_drive(image, description, roi):
    """
    Export all 12 spectral bands as raw Uint16 GeoTIFF to Google Drive.
    Bands: B1,B2,B3,B4,B5,B6,B7,B8,B8A,B9,B11,B12
    """
    task = ee.batch.Export.image.toDrive(
        image          = image.select(OUTPUT_BANDS),
        description    = description,
        folder         = DRIVE_FOLDER,
        fileNamePrefix = description,
        region         = roi,
        scale          = SCALE,
        crs            = 'EPSG:32644',
        fileFormat     = 'GeoTIFF',
        maxPixels      = 1e10,
        formatOptions  = {'cloudOptimized': True},
    )
    task.start()
    return task

# ─────────────────────────────────────────────────────────────────────────────
# SEASONAL COMPOSITE BUILDER
# ─────────────────────────────────────────────────────────────────────────────

def make_seasonal_composite(roi, year, season):
    """Median composite for dry (Nov–Apr) or wet (May–Oct) season."""
    if season == 'dry':
        start, end = f'{year}-11-01', f'{year + 1}-04-30'
    else:
        start, end = f'{year}-05-01', f'{year}-10-31'
    col = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
             .filterBounds(roi)
             .filterDate(start, end)
             .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', MAX_CLOUD_COMPOSITE))
             .select(S2_BANDS + ['QA60'])
             .map(mask_s2_clouds))
    count = col.size().getInfo()
    return col.median().clip(roi), count, start, end

# ─────────────────────────────────────────────────────────────────────────────
# METADATA TRACKING
# ─────────────────────────────────────────────────────────────────────────────

records   = []
all_tasks = []
exported  = 0
skipped   = 0

def log_record(task_id, description, region,
               start, end, count, phase, ctype, extra=''):
    records.append({
        'filename':       f'{description}.tif',
        'drive_folder':   DRIVE_FOLDER,
        'description':    description,
        'region':         region,
        'composite_type': ctype,
        'phase':          phase,
        'start_date':     start,
        'end_date':       end,
        'scene_count':    count,
        'task_id':        task_id,
        'scale_m':        SCALE,
        'bands':          'B1,B2,B3,B4,B5,B6,B7,B8,B8A,B9,B11,B12 (raw Uint16)',
        'crs':            'EPSG:32644',
        'extra_info':     extra,
        'exported_at':    datetime.now().isoformat(),
    })

def save_csv():
    if not records:
        return
    p = f'{OUTPUT_DIR}/metadata/export_log.csv'
    with open(p, 'w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=records[0].keys())
        w.writeheader()
        w.writerows(records)
    print(f"✓ CSV     → {p}  ({len(records)} records)")

def save_manifest(potential):
    manifest = {
        'dataset':          'Hasdeo 4000 GeoTIFF Dataset — Individual Scenes + Composites',
        'generated_at':     datetime.now().isoformat(),
        'drive_folder':     DRIVE_FOLDER,
        'local_metadata':   OUTPUT_DIR,
        'regions':          list(REGIONS.keys()),
        'timeline':         '2015-01-01 – 2024-12-31',
        'strategy':         'Individual scenes (40% cloud) + Monthly composites + Seasonal composites',
        'phases': {
            'pre_event':    '2015–2017',
            'first_event':  '2018–2023',
            'second_event': '2022-03 – 2022-04',
            'post_event':   '2024',
        },
        'scale_m':          SCALE,
        'crs':              'EPSG:32644',
        'bands':            'B1,B2,B3,B4,B5,B6,B7,B8,B8A,B9,B11,B12 (raw Uint16)',
        'summary': {
            'potential_images': potential,
            'tasks_queued':     exported,
            'skipped':          skipped,
            'estimated_gb':     f'{exported * 175 / 1024:.1f} GB  (at ~175 MB avg)',
        },
        'monitor_url':     'https://code.earthengine.google.com/tasks',
        'tasks': [
            {'id': t.id, 'description': t.config.get('description', '')}
            for t in all_tasks if t
        ],
    }
    p = f'{OUTPUT_DIR}/metadata/manifest.json'
    with open(p, 'w') as f:
        json.dump(manifest, f, indent=2)
    print(f"✓ Manifest → {p}")

def save_task_ids():
    p = f'{OUTPUT_DIR}/logs/task_ids.json'
    with open(p, 'w') as f:
        json.dump(
            [{'id': t.id, 'description': t.config.get('description', '')}
             for t in all_tasks if t],
            f, indent=2,
        )
    print(f"✓ Task IDs → {p}")

# ─────────────────────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────────────────────

def main():
    global exported, skipped

    print("=" * 72)
    print("  HASDEO FOREST — 1000–4000 IMAGE INDIVIDUAL SCENES + COMPOSITES")
    print("=" * 72)
    print(f"  Drive folder    : {DRIVE_FOLDER}/")
    print(f"  Cloud limit     : {MAX_CLOUD_INDIVIDUAL}% (individual), {MAX_CLOUD_COMPOSITE}% (composites)")
    print(f"  Resolution      : {SCALE} m")
    print(f"  Bands           : B1–B12 excl. B10  (raw Uint16, 12 bands)")
    print(f"  Timeline        : Jan 2015 – Dec 2024  (10 years)")
    print(f"  Regions         : {len(REGIONS)} → {list(REGIONS.keys())}")
    print()
    print("  PARTS:")
    print("    Part 1 → Individual Sentinel-2 scenes (~5 day revisit, 40% cloud)")
    print("    Part 2 → Monthly median composites (all 120 months)")
    print("    Part 3 → Seasonal composites (dry + wet per year)")
    print("=" * 72)

    init_ee()
    create_local_dirs()

    # ════════════════════════════════════════════════════════════════════════
    # PART 1 — Individual Scenes (THE MAIN BULK — targeting 1000–3000 images)
    # ════════════════════════════════════════════════════════════════════════
    print("\n[ PART 1 ]  Individual Sentinel-2 Scenes  (2015-01 – 2024-12)")
    print("            Cloud cover ≤ 40% | All repeated/overlapping passes included")
    print("-" * 72)

    for region_name, coords in REGIONS.items():
        roi = ee.Geometry.Rectangle(coords)
        print(f"\n  🌳 Region: {region_name}")

        # Fetch all scene metadata for this region in one call
        try:
            full_col = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
                          .filterBounds(roi)
                          .filterDate('2015-01-01', '2024-12-31')
                          .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', MAX_CLOUD_INDIVIDUAL))
                          .select(S2_BANDS + ['QA60']))

            total_scenes = full_col.size().getInfo()
            print(f"    Found {total_scenes} scenes at ≤{MAX_CLOUD_INDIVIDUAL}% cloud cover")

            # Get scene list with metadata
            scene_list = full_col.toList(full_col.size())

            for i in tqdm(range(total_scenes),
                          desc=f"  {region_name[:20]} scenes",
                          unit='scene'):
                try:
                    img = ee.Image(scene_list.get(i))

                    # Get scene date and cloud info
                    date_ms    = img.get('system:time_start').getInfo()
                    cloud_pct  = img.get('CLOUDY_PIXEL_PERCENTAGE').getInfo()
                    scene_id   = img.get('system:index').getInfo()

                    dt         = datetime.utcfromtimestamp(date_ms / 1000)
                    year       = dt.year
                    month      = dt.month
                    date_str   = dt.strftime('%Y%m%d')
                    phase      = get_phase(year, month)

                    # Shorten scene_id to last segment (orbit/tile ref)
                    orbit_tag  = scene_id.split('_')[-1][:8] if scene_id else f'{i:04d}'

                    # e.g. SCENE_Hasdeo_Full_20220315_T44QKF_second_event
                    desc = (f'SCENE_{region_name}_{date_str}_{orbit_tag}_{phase}')[:90]

                    # Apply cloud mask before export
                    masked = mask_s2_clouds(img).clip(roi)

                    task = export_to_drive(masked, desc, roi)
                    all_tasks.append(task)
                    log_record(
                        task.id, desc, region_name,
                        dt.strftime('%Y-%m-%d'), dt.strftime('%Y-%m-%d'),
                        1, phase, 'individual_scene',
                        f'cloud={cloud_pct:.1f}%, orbit={orbit_tag}'
                    )
                    exported += 1
                    time.sleep(0.3)   # gentle rate-limiting

                except Exception as e:
                    skipped += 1
                    # Don't print every skip — only if tqdm isn't hiding it
                    pass

        except Exception as e:
            print(f"    ✗ Failed to list scenes for {region_name}: {str(e)[:80]}")

    print(f"\n  ✓ Part 1 complete — {exported} individual scene tasks queued so far")

    # ════════════════════════════════════════════════════════════════════════
    # PART 2 — Monthly Composites  (Jan 2015 – Dec 2024)
    # ════════════════════════════════════════════════════════════════════════
    print("\n\n[ PART 2 ]  Monthly Median Composites  (Jan 2015 – Dec 2024)")
    print(f"            Cloud cover ≤ {MAX_CLOUD_COMPOSITE}% per scene before compositing")
    print("-" * 72)

    part2_exported = 0
    part2_skipped  = 0

    for region_name, coords in REGIONS.items():
        roi = ee.Geometry.Rectangle(coords)
        print(f"\n  🌳 Region: {region_name}")

        for start, end, phase, year, month in tqdm(
                TIME_PERIODS,
                desc=f"  {region_name[:20]}",
                unit='month'):
            try:
                col = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
                         .filterBounds(roi)
                         .filterDate(start, end)
                         .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', MAX_CLOUD_COMPOSITE))
                         .select(S2_BANDS + ['QA60'])
                         .map(mask_s2_clouds))
                count = col.size().getInfo()

                if count < MIN_IMAGES_COMPOSITE:
                    part2_skipped += 1
                    skipped += 1
                    continue

                composite = col.median().clip(roi)
                desc = f'MON_{region_name}_{year}_{month:02d}_{phase}'[:90]

                task = export_to_drive(composite, desc, roi)
                all_tasks.append(task)
                log_record(task.id, desc, region_name,
                           start, end, count, phase, 'monthly_composite')
                exported  += 1
                part2_exported += 1
                time.sleep(0.3)

            except Exception as e:
                part2_skipped += 1
                skipped += 1

    print(f"\n  ✓ Part 2 complete — {part2_exported} monthly composite tasks queued")

    # ════════════════════════════════════════════════════════════════════════
    # PART 3 — Seasonal Composites  (dry + wet, 2015–2024)
    # ════════════════════════════════════════════════════════════════════════
    print("\n\n[ PART 3 ]  Seasonal Composites  (dry Nov–Apr + wet May–Oct)")
    print("-" * 72)

    part3_exported = 0
    part3_skipped  = 0

    for region_name, coords in REGIONS.items():
        roi = ee.Geometry.Rectangle(coords)
        print(f"\n  🌳 Region: {region_name}")

        for year in range(2015, 2025):
            for season in ('dry', 'wet'):
                try:
                    composite, count, s_date, e_date = make_seasonal_composite(
                        roi, year, season)

                    if count < MIN_IMAGES_COMPOSITE:
                        print(f"    ⊙ {season} {year}: no scenes — skipped")
                        part3_skipped += 1
                        skipped += 1
                        continue

                    if year <= 2017:
                        phase = 'pre_event'
                    elif year == 2022 and season == 'dry':
                        phase = 'second_event'
                    elif year <= 2023:
                        phase = 'first_event'
                    else:
                        phase = 'post_event'

                    desc = f'SEAS_{season.upper()}_{region_name}_{year}_{phase}'[:90]

                    task = export_to_drive(composite, desc, roi)
                    all_tasks.append(task)
                    log_record(task.id, desc, region_name,
                               s_date, e_date, count, phase,
                               f'seasonal_{season}')
                    exported  += 1
                    part3_exported += 1
                    print(f"    ✓ {season.upper()} {year}: {count} scenes → {desc}")
                    time.sleep(0.3)

                except Exception as e:
                    print(f"    ✗ {season} {year}: {str(e)[:70]}")
                    part3_skipped += 1
                    skipped += 1

    print(f"\n  ✓ Part 3 complete — {part3_exported} seasonal composite tasks queued")

    # ════════════════════════════════════════════════════════════════════════
    # SUMMARY
    # ════════════════════════════════════════════════════════════════════════
    potential = (len(REGIONS) * 10 * 70) + (120 * len(REGIONS)) + (10 * len(REGIONS) * 2)

    print("\n" + "=" * 72)
    print("  EXPORT SUMMARY")
    print("=" * 72)
    print(f"  ✓ GEE tasks queued  : {exported} images")
    print(f"    ├─ Individual scenes : {exported - part2_exported - part3_exported}")
    print(f"    ├─ Monthly composites: {part2_exported}")
    print(f"    └─ Seasonal composites: {part3_exported}")
    print(f"  ⊙ Skipped           : {skipped}  (no cloud-free scenes in period)")
    print(f"  📦 Est. total size  : ~{exported * 175 // 1024} GB  (at ~175 MB avg)")
    print()
    print(f"  📁 Drive folder     : {DRIVE_FOLDER}/")
    print(f"  🔗 Monitor tasks    : https://code.earthengine.google.com/tasks")
    print()
    print("  HOW TO GET FILES ON YOUR LOCAL PC:")
    print("  ───────────────────────────────────────────────────────────────")
    print("  1. Wait for GEE tasks to complete (check monitor link above)")
    print("     GEE processes ~3000 tasks; allow 12–48 hrs for completion")
    print("  2. Open Google Drive in browser — files appear in:")
    print(f"     My Drive / {DRIVE_FOLDER}/")
    print("  3. Google Drive desktop app syncs them to your PC automatically")
    print("     Default local path (Windows): C:/Users/<you>/Google Drive/")
    print("                         (Mac)   : ~/Google Drive/")
    print()
    print("  FILE NAMING GUIDE:")
    print("  ───────────────────────────────────────────────────────────────")
    print("  SCENE_<Region>_<YYYYMMDD>_<orbit>_<phase>.tif  ← individual pass")
    print("  MON_<Region>_<YYYY>_<MM>_<phase>.tif           ← monthly composite")
    print("  SEAS_<DRY|WET>_<Region>_<YYYY>_<phase>.tif     ← seasonal composite")
    print()
    print("  NOTE: Files are raw 12-band Uint16 GeoTIFFs.")
    print("  Open with QGIS, rasterio, or any GIS tool.")
    print("  To view as RGB in QGIS: set B4=Red, B3=Green, B2=Blue.")
    print("=" * 72)

    save_csv()
    save_manifest(potential)
    save_task_ids()

    print(f"\n  ✅  Script complete.  {exported} tasks submitted to GEE.")
    print("      Individual scenes will be the bulk of your dataset.")
    print("      Files will appear in Google Drive as tasks finish.\n")


if __name__ == '__main__':
    main()

  HASDEO FOREST — 1000–4000 IMAGE INDIVIDUAL SCENES + COMPOSITES
  Drive folder    : Hasdeo_1000_Dataset_AllBands_new/
  Cloud limit     : 40% (individual), 20% (composites)
  Resolution      : 30 m
  Bands           : B1–B12 excl. B10  (raw Uint16, 12 bands)
  Timeline        : Jan 2015 – Dec 2024  (10 years)
  Regions         : 7 → ['Hasdeo_Full', 'Hasdeo_North', 'Hasdeo_South', 'Hasdeo_Core', 'Hasdeo_East', 'Hasdeo_West', 'Hasdeo_Buffer']

  PARTS:
    Part 1 → Individual Sentinel-2 scenes (~5 day revisit, 40% cloud)
    Part 2 → Monthly median composites (all 120 months)
    Part 3 → Seasonal composites (dry + wet per year)
✓ Earth Engine initialised  |  project: glacier-probe-model-475519
✓ Local dirs ready: hasdeo_4000_local/

[ PART 1 ]  Individual Sentinel-2 Scenes  (2015-01 – 2024-12)
            Cloud cover ≤ 40% | All repeated/overlapping passes included
------------------------------------------------------------------------

  🌳 Region: Hasdeo_Full
    Found 1119 scenes at

  Hasdeo_Full scenes:  11%|█▎           | 118/1119 [25:44<3:38:21, 13.09s/scene]


KeyboardInterrupt: 